# Inverse PINN for Four-Parameter Identification with Fixed Diffusion

This notebook implements an **inverse Physics-Informed Neural Network (PINN)** for the malaria transmission PDE system in the case where the **diffusion coefficient** \(d\) is fixed, while the remaining four parameters are identified from data.

## Objective

The goal is to estimate:
- the advection coefficient \(v\),
- the transmission rate \(\Lambda\),
- the clinical disease probability \(\phi\),
- the treatment coverage \(f_T\),

while keeping:
- \(d\) fixed,
- the epidemiological durations \(d_D, d_T, d_A, d_P\) fixed.

## Main methodological ingredients

This notebook includes:
- a constrained positive parametrization for \(v\),
- a structured OLS projection for the reaction matrix,
- EMA stabilization of OLS estimates,
- analytical proxies extracted from the projected matrix,
- consistency losses for \(\Lambda\), \(\phi\), and \(f_T\),
- a curvature-aware collocation strategy,
- adaptive training phases with task balancing.


## Global setup and reproducibility

This cell initializes the computational environment, sets deterministic seeds, and prepares the runtime configuration for inverse PINN training.

In [ ]:
# Prerequisite: file 'malaria_simulation_results.npz' must be present in the working directory.

import os, math, random, time, json
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F

SEED = 2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# PyTorch determinism (may raise errors if a non-deterministic op is used)
torch.use_deterministic_algorithms(True)

## Configuration and fixed settings

This cell defines the data path, the advection sign convention, the report-only reference values, the fixed diffusion coefficient, and the admissible parameter ranges used during inverse training.

In [ ]:
NPZ_PATH = "malaria_simulation_results.npz"  place the file here
assert os.path.isfile(NPZ_PATH), "Upload 'malaria_simulation_results.npz'"

# Sign convention for advection:
# PDE: Ut + Ua - d Uxx + v Ux - U A^T = 0  -> SIGN_V = +1
# PDE: Ut + Ua - d Uxx - v Ux - U A^T = 0  -> SIGN_V = -1
SIGN_V = -1

# True values (for reporting only; they do not influence training)
TRUE_FOR_REPORT = {
    "d": 0.005,   # FIXED d
    "v": 0.060,
    "Lambda": 0.35,
    "phi": 0.40,
    "f_T": 0.50,
}
d_fixed = TRUE_FOR_REPORT["d"]

# Fixed durations (known parameters)
KNOWN_FIXED = {"d_D": 5.0, "d_T": 5.0, "d_A": 110.0, "d_P": 25.0}

# Ranges of the learned parameters (Λ, φ, f_T) — tighter φ range to avoid upward drift
#RANGES_A3 = {"Lambda": (0.20, 0.55), "phi": (0.30, 0.50), "f_T": (0.20, 0.80)}
RANGES_A3 = {"Lambda": (0.20, 0.55), "phi": (0.20, 0.65), "f_T": (0.20, 0.8)}

# --- Globals for A_ols EMA ---
A_ols_ema = None
EMA_BETA  = 0.985

## Data loading and basic statistics

This cell loads the numerical simulation data, computes the coordinate bounds, standardizes the outputs component-wise, and extracts the initial snapshot used in the inverse setting.

In [ ]:
Z = np.load(NPZ_PATH, allow_pickle=True, mmap_mode='r')
U_snaps = Z["U_snaps"]   # (Nt, Na, Nx, 5)
times   = Z["times"]     # (Nt,)
age_grid= Z["age_grid"]  # (Na,)
xi_grid = Z["xi_grid"]   # (Nx,)
mu_true = Z["mu_true"]  # [d_true, v_true] (if present in the NPZ file)
print("U_snaps:", U_snaps.shape, "| mu_true (NPZ):", mu_true.tolist())

t_min, t_max = float(times.min()), float(times.max())
a_min, a_max = float(age_grid.min()), float(age_grid.max())
x_min, x_max = float(xi_grid.min()), float(xi_grid.max())

# Standardization (component-wise), RAM-friendly
it = np.linspace(0, len(times)-1, min(20, len(times)), dtype=int)
ia = np.linspace(0, len(age_grid)-1, min(40, len(age_grid)), dtype=int)
ix = np.linspace(0, len(xi_grid)-1, min(40, len(xi_grid)), dtype=int)
buf = np.concatenate([U_snaps[i][np.ix_(ia, ix)].reshape(-1,5) for i in it], axis=0)
y_mean = torch.tensor(buf.mean(0), dtype=torch.float32, device=device)
y_std  = torch.tensor(buf.std(0)+1e-6, dtype=torch.float32, device=device)

# Component-wise weighting to balance the 5 channels in the losses
comp_w = (1.0 / (y_std + 1e-6)).to(device)   # (5,)

phi0_np = U_snaps[0]  # snapshot at t=0

d_true_npz = float(mu_true[0])
v_true_npz = float(mu_true[1])
print(f"NPZ truth: d={d_true_npz:.6f}, v={v_true_npz:.6f}")
print(f"Report truth (config): {TRUE_FOR_REPORT}")

## Local finite-difference derivatives from data

This cell builds finite-difference approximations of the local time, age, and exposure derivatives directly from the numerical data, with stronger smoothing along the exposure axis.

In [ ]:
def _sample_interior_indices(n, margin=2):
    Nt, Na, Nx, _ = U_snaps.shape
    ti = np.random.randint(margin, Nt-margin, size=n)
    ai = np.random.randint(margin, Na-margin, size=n)
    xi = np.random.randint(margin, Nx-margin, size=n)
    return ti, ai, xi

def _five_point_first(m2, m1, c0, p1, p2, h):
    return (-p2 + 8*p1 - 8*m1 + m2) / (12.0*h)

def _five_point_second(m2, m1, c0, p1, p2, h):
    return (-p2 + 16*p1 - 30*c0 + 16*m1 - m2) / (12.0*h*h)

def _smooth_along_x(arr_1d, w=9):
    if w <= 1: return arr_1d
    pad = w//2
    pad_left  = arr_1d[:pad][::-1]
    pad_right = arr_1d[-pad:][::-1]
    ext = np.concatenate([pad_left, arr_1d, pad_right], axis=0)
    ker = np.ones(w, dtype=np.float64)/w
    sm = np.convolve(ext, ker, mode='valid')
    return sm[:len(arr_1d)]

def _data_local_derivs(n_samples):
    Nt, Na, Nx, C = U_snaps.shape
    ti, ai, xi = _sample_interior_indices(n_samples, margin=2)

    dt = float(times[1]-times[0]) if Nt>1 else 1.0
    da = float(age_grid[1]-age_grid[0]) if Na>1 else 1.0
    dx = float(xi_grid[1]-xi_grid[0]) if Nx>1 else 1.0

    Ut_list, Ua_list, Ux_list, Uxx_list, U_list = [], [], [], [], []
    for k in range(n_samples):
        t0, a0, x0 = ti[k], ai[k], xi[k]

        Ut_m2 = U_snaps[t0-2, a0,   x0,   :]
        Ut_m1 = U_snaps[t0-1, a0,   x0,   :]
        Ut_0  = U_snaps[t0,   a0,   x0,   :]
        Ut_p1 = U_snaps[t0+1, a0,   x0,   :]
        Ut_p2 = U_snaps[t0+2, a0,   x0,   :]

        Ua_m2 = U_snaps[t0,   a0-2, x0,   :]
        Ua_m1 = U_snaps[t0,   a0-1, x0,   :]
        Ua_0  = U_snaps[t0,   a0,   x0,   :]
        Ua_p1 = U_snaps[t0,   a0+1, x0,   :]
        Ua_p2 = U_snaps[t0,   a0+2, x0,   :]

        line_x   = U_snaps[t0, a0, :, :]
        line_x_s = np.stack([_smooth_along_x(line_x[:,c], w=9) for c in range(5)], axis=1)
        Ux_m2 = line_x_s[x0-2, :]; Ux_m1 = line_x_s[x0-1, :]; Ux_0 = line_x_s[x0, :]
        Ux_p1 = line_x_s[x0+1, :]; Ux_p2 = line_x_s[x0+2, :]

        Ut_k  = _five_point_first (Ut_m2, Ut_m1, Ut_0, Ut_p1, Ut_p2, dt)
        Ua_k  = _five_point_first (Ua_m2, Ua_m1, Ua_0, Ua_p1, Ua_p2, da)
        Ux_k  = _five_point_first (Ux_m2, Ux_m1, Ux_0, Ux_p1, Ux_p2, dx)
        Uxx_k = _five_point_second(Ux_m2, Ux_m1, Ux_0, Ux_p1, Ux_p2, dx)

        Ut_list.append(Ut_k); Ua_list.append(Ua_k); Ux_list.append(Ux_k); Uxx_list.append(Uxx_k)
        U_list.append(Ux_0)

    Ut_t  = torch.tensor(np.stack(Ut_list,0),  dtype=torch.float32, device=device)
    Ua_t  = torch.tensor(np.stack(Ua_list,0),  dtype=torch.float32, device=device)
    Ux_t  = torch.tensor(np.stack(Ux_list,0),  dtype=torch.float32, device=device)
    Uxx_t = torch.tensor(np.stack(Uxx_list,0), dtype=torch.float32, device=device)
    U_t   = torch.tensor(np.stack(U_list,0),   dtype=torch.float32, device=device)
    return Ut_t, Ua_t, Ux_t, Uxx_t, U_t

## OLS estimators and data-only losses

This cell defines OLS-based estimators for the advection coefficient and the projected reaction matrix, using only local derivative information extracted from data.

In [ ]:
@torch.no_grad()
def theta_ols_from_data_v_only(batch_n=8000, ridge=2e-4, A_use=None, d_fix=0.005):
    Ut, Ua, Ux, Uxx, U = _data_local_derivs(batch_n)
    if A_use is None:
        raise ValueError("theta_ols_from_data_v_only requires A_use.")
    S = Ut + Ua - (U @ A_use.T) - d_fix*Uxx

    v_list = []
    for k in range(5):
        X = torch.stack([-(SIGN_V * Ux[:,k]), torch.ones_like(Ux[:,k])], dim=1)
        Y = S[:,k:k+1]
        mu = X.mean(0, keepdim=True); sg = X.std(0, keepdim=True) + 1e-8
        Xn = (X - mu)/sg
        beta_n = torch.linalg.lstsq(Xn, Y).solution
        beta = beta_n / sg.T
        v_k = -beta[0,0].item()
        v_list.append(v_k)
    q1, q3 = np.percentile(v_list, [25, 75]); iqr=q3-q1; lo,hi=q1-1.5*iqr, q3+1.5*iqr
    vv=[x for x in v_list if lo<=x<=hi]; vv=vv if len(vv)>=1 else v_list
    return float(np.median(vv))

@torch.no_grad()
def A_ols_from_data(d_use, v_use, batch_n=6144, ridge=2e-4, max_tries=4):
    Ut, Ua, Ux, Uxx, U = _data_local_derivs(batch_n)
    dtype = torch.float64
    U   = U.to(dtype)
    Ut  = Ut.to(dtype); Ua = Ua.to(dtype); Ux = Ux.to(dtype); Uxx = Uxx.to(dtype)

    RHS = Ut + Ua - d_use*Uxx + SIGN_V*v_use*Ux

    stdU = U.std(dim=0, keepdim=True).clamp_min(1e-8)
    U_s  = U / stdU

    I5 = torch.eye(5, dtype=dtype, device=U.device)
    zero5 = torch.zeros(5, dtype=dtype, device=U.device)

    W_scaled = torch.empty((5,5), dtype=dtype, device=U.device)
    for k in range(5):
        y = RHS[:, k]
        this_ridge = ridge
        sol = None
        for _ in range(max_tries):
            U_aug = torch.vstack([U_s, (this_ridge**0.5) * I5])
            y_aug = torch.hstack([y, zero5])
            sol = torch.linalg.lstsq(U_aug, y_aug).solution
            if torch.isfinite(sol).all():
                break
            this_ridge *= 10.0
        if sol is None or not torch.isfinite(sol).all():
            U_pinv = torch.linalg.pinv(U_aug)
            sol = U_pinv @ y_aug
        W_scaled[:, k] = sol

    A_ols_T = (W_scaled / stdU.squeeze(0))
    A_ols   = A_ols_T.T
    return A_ols.to(torch.float32)

## Auxiliary losses and utility functions

This cell gathers helper loss functions, robust penalties, structured matrix matching terms, and the reconstructed inflow function \(b(t)\).

In [ ]:
def huber(x, delta=0.05):
    a = x.abs()
    quad = torch.minimum(a, torch.tensor(delta, device=x.device))
    lin  = a - quad
    return 0.5*(quad**2)/delta + lin

def A_ols_focus_huber(A_cur, A_ols, delta=0.01, w_focus=4.0):
    D = A_cur - A_ols
    absD = D.abs()
    L_h = torch.where(absD <= delta, 0.5*(absD**2), delta*(absD - 0.5*delta)).mean()
    key = [(0,0),(1,0),(2,0),(3,3)]
    Lf = 0.0
    for (i,j) in key:
        dij = absD[i,j]
        lij = (0.5*dij**2) if dij <= delta else (delta*(dij - 0.5*delta))
        Lf = Lf + lij
    Lf = Lf / len(key)
    return L_h + w_focus*Lf

def A_focus_loss(A_cur, A_tgt, w_entries=(1.0,1.0,1.0,0.5), delta=0.05):
    Ls = []
    Ls.append(w_entries[0]*huber(A_cur[0,0]-A_tgt[0,0], delta).mean())
    Ls.append(w_entries[1]*huber(A_cur[1,0]-A_tgt[1,0], delta).mean())
    Ls.append(w_entries[2]*huber(A_cur[2,0]-A_tgt[2,0], delta).mean())
    Ls.append(w_entries[3]*huber(A_cur[3,3]-A_tgt[3,3], delta).mean())
    return sum(Ls)

def b_from_npz(tval: float):
    if tval <= times[0]: return float(U_snaps[0,0,:,0].mean())
    if tval >= times[-1]: return float(U_snaps[-1,0,:,0].mean())
    i = max(0, min(len(times)-2, int(np.searchsorted(times, tval)-1)))
    w = (tval-times[i])/(times[i+1]-times[i]+1e-12)
    return float((1-w)*U_snaps[i,0,:,0].mean() + w*U_snaps[i+1,0,:,0].mean())

## Learnable inverse parameters

This cell defines the constrained parameterizations used to learn the positive advection coefficient \(v\) and the three reaction parameters \(\Lambda\), \(\phi\), and \(f_T\).

In [ ]:
def _inv_sig_from_val(val, lo, hi):
    x = float(np.clip((val-lo)/(hi-lo), 1e-6, 1-1e-6))
    return math.log(x/(1-x))

class LearnV(nn.Module):
    def __init__(self, v_lo=1e-4, v_hi=0.25, v_init=0.06):
        super().__init__()
        self.v_lo, self.v_hi = v_lo, v_hi
        x0 = float(np.clip((v_init - v_lo)/(v_hi - v_lo), 1e-6, 1-1e-6))
        self.r_v = nn.Parameter(torch.tensor(math.log(x0/(1.0 - x0)), dtype=torch.float32))
    def forward(self):
        s = torch.sigmoid(self.r_v)
        return self.v_lo + (self.v_hi - self.v_lo) * s

class LearnA3(nn.Module):
    def __init__(self, p_init, ranges):
        super().__init__()
        self.keys = ["Lambda","phi","f_T"]
        rho=[]
        for k in self.keys:
            lo,hi = ranges[k]
            rho.append(_inv_sig_from_val(p_init[k], lo,hi))
        self.rho = nn.Parameter(torch.tensor(rho, dtype=torch.float32))
        self.lo  = torch.tensor([ranges[k][0] for k in self.keys], device=device)
        self.hi  = torch.tensor([ranges[k][1] for k in self.keys], device=device)
    def forward(self):
        s = torch.sigmoid(self.rho.to(device))
        p = self.lo + (self.hi - self.lo)*s
        return {k: p[i] for i,k in enumerate(self.keys)}

def A_from_parts(learned, fixed):
    La, ph, fT = learned["Lambda"], learned["phi"], learned["f_T"]
    dD, dT, dA, dP = [torch.tensor(fixed[k], device=device) for k in ["d_D","d_T","d_A","d_P"]]
    A = torch.zeros(5,5,device=device)
    A[0,0]=-La; A[0,3]=1/dA; A[0,4]=1/dP
    A[1,0]=ph*(1-fT)*La; A[1,1]=-1/dD; A[1,3]=ph*(1-fT)*La
    A[2,0]=ph*fT*La; A[2,2]=-1/dT; A[2,3]=ph*fT*La
    A[3,0]=(1-ph)*La; A[3,1]=1/dD; A[3,3]=-(ph*La+1/dA)
    A[4,2]=1/dT; A[4,4]=-1/dP
    return A

## Neural network architecture

This cell defines the Fourier-feature encoding and the MLP used by the inverse PINN to approximate the solution field.

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self,K=4): super().__init__(); self.K=K
    def forward(self,z):
        feats=[z]
        for f in range(1,self.K+1):
            w=2*math.pi*f; feats += [torch.sin(w*z), torch.cos(w*z)]
        return torch.cat(feats,1)

class EncodedMLP(nn.Module):
    def __init__(self, width=64, depth=5, act="tanh", K=4):
        super().__init__()
        self.enc=FourierFeatures(K)
        in_dim=3*(1+2*K)
        actl={"tanh":nn.Tanh(),"relu":nn.ReLU(),"silu":nn.SiLU()}[act]
        layers=[nn.Linear(in_dim,width),actl]
        for _ in range(depth-1): layers+=[nn.Linear(width,width),actl]
        layers+=[nn.Linear(width,5)]
        self.net=nn.Sequential(*layers)
    def forward(self,x):
        t_,a_,x_ = x[:, :1], x[:,1:2], x[:,2:3]
        return self.net(torch.cat([self.enc(t_), self.enc(a_), self.enc(x_)],1))

model   = EncodedMLP(64,5,"tanh",K=4).to(device)
v_learn = LearnV(v_init=0.06).to(device)

def predict_std(t_, a_, x_): return model(torch.cat([t_, a_, x_], 1))

## Sampling utilities

This cell defines the different batches used during training: data points, initial condition samples, inflow samples, and physics collocation points.

In [ ]:
def pick_grid(n):
    it=np.random.randint(0,len(times),size=n)
    ia=np.random.randint(0,len(age_grid),size=n)
    ix=np.random.randint(0,len(xi_grid),size=n)
    t=torch.tensor((times[it]-t_min)/(t_max-t_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    a=torch.tensor((age_grid[ia]-a_min)/(a_max-a_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    x=torch.tensor((xi_grid[ix]-x_min)/(x_max-x_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    U=torch.tensor(U_snaps[it,ia,ix,:],dtype=torch.float32,device=device)
    return t,a,x,(U-y_mean)/y_std

def batch_IC(n):
    ia=np.random.randint(0,len(age_grid),size=n)
    ix=np.random.randint(0,len(xi_grid),size=n)
    t0=torch.zeros(n,1,device=device)
    a=torch.tensor((age_grid[ia]-a_min)/(a_max-a_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    x=torch.tensor((xi_grid[ix]-x_min)/(x_max-x_min+1e-12),dtype=torch.float32,device=device).reshape(-1,1)
    U=torch.tensor(phi0_np[ia,ix,:],dtype=torch.float32,device=device)
    return t0,a,x,(U-y_mean)/y_std

def batch_A0(n):
    t=torch.rand(n,1,device=device); a0=torch.zeros_like(t); x=torch.rand(n,1,device=device)
    t_phys=(t*(t_max-t_min)+t_min).flatten().tolist()
    b_vals=torch.tensor([b_from_npz(tt) for tt in t_phys],dtype=torch.float32,device=device).reshape(-1,1)
    U=torch.cat([b_vals, torch.zeros_like(b_vals).repeat(1,4)],1)
    return t,a0,x,(U-y_mean)/y_std

def batch_phys(n, edge_bias=0.6):
    nb=int(n*edge_bias); nu=n-nb
    def beta_edge(m):
        z=torch.distributions.Beta(0.5,0.5).sample((m,1)).to(device); return z.clamp(0,1)
    t_b=beta_edge(nb); a_b=beta_edge(nb)
    k=nb//2; x_b=torch.cat([torch.zeros(k,1,device=device), torch.ones(nb-k,1,device=device)],0)
    t_u=torch.rand(nu,1,device=device); a_u=torch.rand(nu,1,device=device); x_u=torch.rand(nu,1,device=device)
    t=torch.cat([t_b,t_u],0); a=torch.cat([a_b,a_u],0); x=torch.cat([x_b,x_u],0)
    idx=torch.randperm(n,device=device)
    return t[idx],a[idx],x[idx]

def batch_phys_curvature_aware(n, pool=6, edge_bias=0.8):
    m = n*pool
    t,a,x = batch_phys(m, edge_bias=edge_bias)
    eps = 1.0 / max(1, (len(xi_grid)-1))
    with torch.no_grad():
        U0  = model(torch.cat([t, a, x.clamp(0,1)], 1))
        Up  = model(torch.cat([t, a, (x+eps).clamp(0,1)], 1))
        Um  = model(torch.cat([t, a, (x-eps).clamp(0,1)], 1))
        Uxx = (Up - 2*U0 + Um) / (eps**2)
        Uxx = Uxx * (y_std / ((x_max-x_min+1e-12)**2))
        score = Uxx.abs().mean(dim=1)
    idx = torch.topk(score, k=n, largest=True).indices
    return t[idx], a[idx], x[idx]

## PDE losses and physics scaling

This cell defines the PDE-on-data loss, the second-derivative matching loss, physics-aware scaling factors, and the normalized PDE residual used during inverse training.

In [ ]:
def pde_on_data_loss(n_samples=4096, A3=None, v_val=None):
    Ut_t, Ua_t, Ux_t, Uxx_t, U_t = _data_local_derivs(n_samples)
    d = torch.tensor(0.005, device=device)
    v = v_val if v_val is not None else v_learn()
    A = A_from_parts(A3 if A3 is not None else A3_learn(), KNOWN_FIXED)
    rhs  = Ut_t + Ua_t - d*Uxx_t + SIGN_V * v * Ux_t
    lhs  = U_t @ A.T
    res  = (lhs - rhs) * comp_w
    return (res.pow(2).mean())

def uxx_match_loss(n_samples=4096):
    _, _, _, Uxx_data, _ = _data_local_derivs(n_samples)
    t = torch.rand(n_samples,1,device=device)
    a = torch.rand(n_samples,1,device=device)
    x = torch.rand(n_samples,1,device=device)
    eps = 1.0 / max(1, (len(xi_grid)-1))
    Up  = model(torch.cat([t, a, (x+eps).clamp(0,1)], 1))
    U0  = model(torch.cat([t, a, x], 1))
    Um  = model(torch.cat([t, a, (x-eps).clamp(0,1)], 1))
    x_rng=(x_max-x_min)+1e-12
    Uxx_mod = (Up - 2*U0 + Um) / (eps**2) * (y_std / (x_rng**2))
    diff = (Uxx_mod - Uxx_data.to(device)) * comp_w
    return (diff**2).mean()

@torch.no_grad()
def compute_physics_scales_inv(n0: int = 2000):
    t = torch.rand(n0,1,device=device); a = torch.rand(n0,1,device=device); x = torch.rand(n0,1,device=device)
    with torch.enable_grad():
        t_rng=(t_max-t_min)+1e-12; a_rng=(a_max-a_min)+1e-12; x_rng=(x_max-x_min)+1e-12
        t1=t.clone().requires_grad_(True); a1=a; x1=x
        Ustd_t=model(torch.cat([t1,a1,x1],1))
        Ut=torch.cat([torch.autograd.grad(Ustd_t[:,k].sum(), t1, create_graph=True, retain_graph=(k<4))[0] for k in range(5)], 1)*(y_std/t_rng)
        t2=t; a2=a.clone().requires_grad_(True); x2=x
        Ustd_a=model(torch.cat([t2,a2,x2],1))
        Ua=torch.cat([torch.autograd.grad(Ustd_a[:,k].sum(), a2, create_graph=True, retain_graph=(k<4))[0] for k in range(5)], 1)*(y_std/a_rng)
        t3=t; a3=a; x3=x
        Ustd_c=model(torch.cat([t3,a3,x3],1))
        U=Ustd_c*y_std+y_mean
        eps=1.0/max(1,(len(xi_grid)-1))
        Up=model(torch.cat([t3,a3,(x3+eps).clamp(0,1)],1))
        Um=model(torch.cat([t3,a3,(x3-eps).clamp(0,1)],1))
        Ux =(Up-Um)/(2*eps)*(y_std/x_rng)
        Uxx=(Up-2*Ustd_c+Um)/(eps**2)*(y_std/(x_rng**2))
    AU = U @ A_from_parts(A3_learn(), KNOWN_FIXED).T
    def med(v): return torch.quantile(v.abs(), 0.5, dim=0).clamp_min(1e-6)
    return {"Ut":med(Ut), "Ua":med(Ua), "Ux":med(Ux), "Uxx":med(Uxx), "AU":med(AU)}

def pde_residual_only_scaled(t_, a_, x_, scales):
    d = torch.tensor(0.005, device=device)
    v = v_learn()
    A = A_from_parts(A3_learn(), KNOWN_FIXED)
    t1=t_.clone().requires_grad_(True); a1=a_.detach(); x1=x_.detach()
    Ustd_t=model(torch.cat([t1,a1,x1],1))
    Ut = torch.cat([torch.autograd.grad(Ustd_t[:,k].sum(), t1, create_graph=True, retain_graph=True)[0] for k in range(5)], 1)
    t2=t_.detach(); a2=a_.clone().requires_grad_(True); x2=x_.detach()
    Ustd_a=model(torch.cat([t2,a2,x2],1))
    Ua = torch.cat([torch.autograd.grad(Ustd_a[:,k].sum(), a2, create_graph=True, retain_graph=True)[0] for k in range(5)], 1)
    t3=t_.detach(); a3=a_.detach(); x3=x_.detach()
    Ustd_c=model(torch.cat([t3,a3,x3],1))
    U=Ustd_c*y_std+y_mean
    eps=1.0/max(1,(len(xi_grid)-1))
    Up=model(torch.cat([t3,a3,(x3+eps).clamp(0,1)],1))
    Um=model(torch.cat([t3,a3,(x3-eps).clamp(0,1)],1))
    x_rng=(x_max-x_min)+1e-12
    Ux =(Up-Um)/(2*eps)*(y_std/x_rng)
    Uxx=(Up-2*Ustd_c+Um)/(eps**2)*(y_std/(x_rng**2))
    r = (Ut/scales["Ut"]) + (Ua/scales["Ua"]) - d*(Uxx/scales["Uxx"]) + SIGN_V*v*(Ux/scales["Ux"]) - (U @ A.T)/scales["AU"]
    return r * comp_w

def h1_penalty(n=1024):
    t=torch.rand(n,1,device=device); a=torch.rand(n,1,device=device); x=torch.rand(n,1,device=device).requires_grad_(True)
    Ustd = model(torch.cat([t,a,x],1))
    dUdx=[]
    for k in range(5):
        gk = torch.autograd.grad(Ustd[:,k].sum(), x, create_graph=False, retain_graph=True)[0]
        dUdx.append(gk)
    dUdx=torch.cat(dUdx,1)
    x_rng=(x_max-x_min)+1e-12
    dUdx = dUdx * (y_std/x_rng)
    return (dUdx**2).mean()

## Initialization of inverse parameters

This cell selects an initial parameter guess from data-only candidates and instantiates the learnable inverse parameter block.

In [ ]:
def choose_p_init_from_ranges(ranges, n_cands=24, jitter=0.05, batch_n=3000, ridge=2e-4):
    def mid(lo,hi): return 0.5*(lo+hi)
    base = {k: mid(*ranges[k]) for k in ("Lambda","phi","f_T")}
    A_mid = A_from_parts(
        {"Lambda": torch.tensor(base["Lambda"], device=device),
         "phi":    torch.tensor(base["phi"],    device=device),
         "f_T":    torch.tensor(base["f_T"],    device=device)},
        KNOWN_FIXED
    ).detach()
    Ut, Ua, Ux, Uxx, U = _data_local_derivs(batch_n)
    RHS_base = Ut + Ua - 0.005*Uxx + SIGN_V*0.06*Ux

    cands=[]
    for _ in range(n_cands):
        cand={}
        for k in ("Lambda","phi","f_T"):
            lo,hi = ranges[k]
            span = (hi-lo)
            val  = base[k] + (np.random.rand()-0.5)*(2*jitter*span)
            val  = float(np.clip(val, lo, hi))
            cand[k]=val
        cands.append(cand)
    best=None; best_score=float("inf")
    for c in cands:
        A = A_from_parts(
            {"Lambda": torch.tensor(c["Lambda"],device=device),
             "phi":    torch.tensor(c["phi"],   device=device),
             "f_T":    torch.tensor(c["f_T"],   device=device)},
            KNOWN_FIXED
        )
        res = (U @ A.T) - RHS_base
        sc  = float((res**2).mean().item())
        if sc < best_score:
            best_score = sc; best = c
    return best

P_INIT = choose_p_init_from_ranges(RANGES_A3, n_cands=24, jitter=0.05)
A3_learn    = LearnA3(P_INIT, RANGES_A3).to(device)
print("P_INIT:", P_INIT)

## Analytical proxies and consistency losses

This cell defines the analytical proxies extracted from the OLS matrix, along with ratio-based and prior-based regularization terms used to stabilize inverse identification.

In [ ]:
@torch.no_grad()
def a3_proxies_from_A(A: torch.Tensor):
    eps = 1e-8
    A10 = float(A[1,0].item()); A20 = float(A[2,0].item()); A30 = float(A[3,0].item())
    denom_phi = A10 + A20 + A30
    denom_ft  = A10 + A20
    La_hat = float(-(A[0,0].item()))
    phi_hat = float((A10 + A20) / (denom_phi if abs(denom_phi)>eps else eps))
    fT_hat  = float(A20 / (denom_ft if abs(denom_ft)>eps else eps))
    return La_hat, phi_hat, fT_hat

def ratio_consistency_loss(A_proxy: torch.Tensor, learned: dict,
                           w=(1.0, 8.0, 1.0), delta=0.01):  # stronger weight on φ
    La_hat, phi_hat, fT_hat = a3_proxies_from_A(A_proxy)
    La_pred = learned["Lambda"]; phi_pred = learned["phi"]; fT_pred = learned["f_T"]
    def huber1(x, d=delta):
        ax = torch.abs(x)
        return torch.where(ax <= d, 0.5*(ax**2)/d, ax - 0.5*d)
    LLa  = huber1(La_pred - torch.tensor(La_hat,  device=La_pred.device, dtype=La_pred.dtype))
    Lphi = huber1(phi_pred - torch.tensor(phi_hat, device=La_pred.device, dtype=La_pred.dtype))
    LfT  = huber1(fT_pred - torch.tensor(fT_hat,  device=La_pred.device, dtype=La_pred.dtype))
    return w[0]*LLa + w[1]*Lphi + w[2]*LfT

# Additional loss centered on φ through ratios (optional but useful)
def phi_ratio_triplet_loss(A_proxy: torch.Tensor, A_cur: torch.Tensor, delta=0.01, w_phi=4.0):
    # Compares φ(A_cur) vs φ(A_proxy) only through ratios A10, A20, A30
    _, phi_hat, _ = a3_proxies_from_A(A_proxy)
    _, phi_cur, _ = a3_proxies_from_A(A_cur.detach())
    diff = torch.tensor(phi_cur - phi_hat, device=A_cur.device, dtype=A_cur.dtype)
    ax = diff.abs()
    h = torch.where(ax <= delta, 0.5*(ax**2)/delta, ax - 0.5*delta)
    return w_phi * h

def v_ols_prior_loss_local(d_fixed, A_for_theta, Wv_ols=600.0, batch_n=8000):
    Ut, Ua, Ux, Uxx, U = _data_local_derivs(batch_n)
    S = Ut + Ua - (U @ A_for_theta.T) - d_fixed*Uxx
    v_cands = []
    ridge = 2e-4
    for k in range(5):
        X = torch.stack([Ux[:,k]], dim=1)
        Y = S[:,k:k+1] / (SIGN_V + 1e-8)
        mu = X.mean(0, keepdim=True); sg = X.std(0, keepdim=True) + 1e-8
        Xn = (X - mu)/sg
        beta_n = torch.linalg.solve(Xn.T @ Xn + ridge*torch.eye(1, device=device), Xn.T @ Y)
        beta = beta_n / sg.T
        v_k = beta[0,0].item()
        v_cands.append(v_k)
    v_ols = float(np.median(v_cands))
    v_ols = abs(v_ols)
    v = v_learn()
    return Wv_ols * (v - torch.tensor(v_ols, dtype=v.dtype, device=v.device))**2

## Uncertainty-based multi-task weighting

This cell introduces uncertainty-weighted balancing across the multiple inverse losses used during training.

In [ ]:
class UncertaintyWeights(nn.Module):
    def __init__(self):
        super().__init__()
        self.names = ["DATA","PDE","IC","A0","FLUX","STRUCT","UXX","PDATA","AFOCUS","RATIO","PHIR"]
        self.logsig = nn.ParameterDict({n: nn.Parameter(torch.tensor(0.0,device=device)) for n in self.names})
    def forward(self, losses: dict):
        total = 0.0; eff={}
        for n in self.names:
            if n not in losses: continue
            s=self.logsig[n]; Li=losses[n]
            wi=torch.exp(-s); total = total + wi*Li + s
            eff[n]=float(wi.detach().cpu())
        return total, eff

## Inverse training loop

This cell runs the full inverse training procedure, including adaptive OLS refresh, phase-dependent weights, structured consistency losses, and early stopping.

In [ ]:
BPhys = 6144; BData = 2048; BIC = 1024; BA0 = 4096; BNeu = 4096
EPOCHS = 6000

uw = UncertaintyWeights().to(device)
scales = compute_physics_scales_inv(n0=4000)

params_model = list(model.parameters())
params_v     = list(v_learn.parameters())
params_A     = list(A3_learn.parameters())

opt = torch.optim.AdamW(
    [
      {'params': params_model, 'lr': 1e-3,  'weight_decay': 1e-4},
      {'params': params_v,     'lr': 7e-4,  'weight_decay': 1e-4},
      {'params': params_A,     'lr': 1.2e-4,  'weight_decay': 1e-4},
    ]
)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS*2, eta_min=2e-4)

L_H1 = 2e-4

A_ols_cache = None
v_ols_cache = None

print("Training (inverse, d FIXED, OLS schedule + adaptive batches + early stopping)...")
t0 = time.time()

P1_end = 2500
P2_end = 5000

def scale_grads_local(module, scale):
    for p in module.parameters():
        if p.grad is not None:
            p.grad.mul_((scale if isinstance(scale, (float,int)) else 1.0))

def trainable_params_from(optimizer):
    return [p for g in optimizer.param_groups for p in g['params'] if getattr(p, 'requires_grad', True)]

def cadence_recomp(ep: int) -> int:
    if ep < 800:   return 2
    if ep < 2000:  return 8
    if ep < 3400:  return 15
    if ep < 5200:  return 20
    return 25

def rel_err_pct(h, t):
    return float(abs(h - t) / (abs(t) + 1e-12) * 100.0)

ok_count = 0

for ep in range(1, EPOCHS+1):
    if ep <= P1_end:
        BPhys_eff = BPhys; BNeu_eff  = BNeu
    else:
        BPhys_eff = 3072;   BNeu_eff  = 2048

    # Phase weights (reinforce φ without damaging v and f_T)
    edge_bias = 0.95 if ep < 1200 else (0.9 if ep < 3000 else 0.8)
    if ep <= P1_end:
        phase    = "P1"; W_UXX=10.0; W_STRUCT=1.2; W_AFOCUS=2.0; W_RATIO=2.2; gV, gA = 0.8, 1.2; uxx_ns=4096
    elif ep <= P2_end:
        phase    = "P2"; W_UXX=12.0; W_STRUCT=1.4; W_AFOCUS=1.6; W_RATIO=1.2; gV, gA = 1.0, 0.7; uxx_ns=3072
    else:
        phase    = "P3"; W_UXX=5.0;  W_STRUCT=1.0; W_AFOCUS=1.0; W_RATIO=1.6; gV, gA = 1.0, 1.0; uxx_ns=2048

    # OLS recomputation + structural projection -> EMA
    recomp_every = cadence_recomp(ep)
    if (ep % recomp_every) == 1 or (A_ols_cache is None):
        with torch.no_grad():
            A_for_theta = A_from_parts(A3_learn(), KNOWN_FIXED).detach()
            v_ols = theta_ols_from_data_v_only(batch_n=12000, A_use=A_for_theta, d_fix=0.005)
            v_ols = abs(v_ols)
            v_ols_cache = v_ols
            if ep < 1200:   bn_A = 8192
            elif ep < 3400: bn_A = 4096
            else:           bn_A = 2048
            A_ols = A_ols_from_data(d_use=0.005, v_use=v_ols, batch_n=bn_A, ridge=5e-5)
            # --- Structural projection ---
            La_hat, phi_hat, fT_hat = a3_proxies_from_A(A_ols)
            A_proj = A_from_parts(
                {"Lambda": torch.tensor(La_hat, device=device),
                 "phi":    torch.tensor(phi_hat, device=device),
                 "f_T":    torch.tensor(fT_hat, device=device)},
                KNOWN_FIXED
            ).detach()
            A_ols_cache = A_proj
            if globals().get('A_ols_ema', None) is None:
                globals()['A_ols_ema'] = A_proj.clone()
            else:
                globals()['A_ols_ema'] = EMA_BETA * globals()['A_ols_ema'] + (1.0 - EMA_BETA) * A_proj
    else:
        v_ols = v_ols_cache
        A_for_theta  = A_from_parts(A3_learn(), KNOWN_FIXED).detach()

    opt.zero_grad(set_to_none=True)

    # Curvature-aware physics batches
    t_p, a_p, x_p = batch_phys_curvature_aware(BPhys_eff, pool=8, edge_bias=edge_bias)
    r_pde  = pde_residual_only_scaled(t_p, a_p, x_p, scales)
    L_pde  = (r_pde**2).mean()

    L_pdata = pde_on_data_loss(n_samples=4096)

    def robin_flux_loss(BNeu_eff):
        d_det, v_det = 0.005, float(v_learn().detach())
        t_n = torch.rand(BNeu_eff,1,device=device)
        a_n = torch.rand(BNeu_eff,1,device=device)
        x0 = torch.zeros(BNeu_eff//2,1,device=device, requires_grad=True)
        x1 = torch.ones (BNeu_eff - BNeu_eff//2,1,device=device, requires_grad=True)
        def flux_loss_at_x(xb, t_slice, a_slice):
            Ustd = model(torch.cat([t_slice, a_slice, xb], 1))
            d_list=[]
            for k in range(5):
                gk = torch.autograd.grad(Ustd[:,k].sum(), xb, create_graph=True, retain_graph=True)[0]
                d_list.append(gk)
            dUdx = torch.cat(d_list, 1) * (y_std / ((x_max - x_min) + 1e-12))
            U_phys = Ustd * y_std + y_mean
            F = -d_det * dUdx + SIGN_V * v_det * U_phys
            return (F**2).mean()
        n0 = x0.shape[0]
        L0 = flux_loss_at_x(x0, t_n[:n0], a_n[:n0])
        L1 = flux_loss_at_x(x1, t_n[n0:], a_n[n0:])
        return 0.5*(L0 + L1)
    L_flux = robin_flux_loss(BNeu_eff)

    # Structural losses guided by structured EMA
    A_cur    = A_from_parts(A3_learn(), KNOWN_FIXED)
    L_struct = A_ols_focus_huber(A_cur, globals()['A_ols_ema'], delta=0.01, w_focus=4.0)
    L_afocus = A_focus_loss(A_cur, globals()['A_ols_ema'], w_entries=(1.0,1.0,1.0,0.5), delta=0.05)
    L_ratio  = ratio_consistency_loss(globals()['A_ols_ema'], A3_learn(), w=(1.0, 8.0, 1.0), delta=0.01)
    L_phir   = phi_ratio_triplet_loss(globals()['A_ols_ema'], A_cur, delta=0.01,
                                      w_phi=(12.0 if phase=="P1" else (6.0 if phase=="P2" else 3.0)))

    # Data terms
    t_d, a_d, x_d, Ud = pick_grid(BData); Up_d  = predict_std(t_d,a_d,x_d); L_data = ((Up_d-Ud)**2).mean()
    t_ic, a_ic, x_ic, U0 = batch_IC(BIC); Up_ic = predict_std(t_ic,a_ic,x_ic); L_ic   = ((Up_ic-U0)**2).mean()
    t_a0, a0, x_a0, Ua   = batch_A0(BA0); Up_a0 = predict_std(t_a0,a0,x_a0); L_a0   = ((Up_a0-Ua)**2).mean()

    L_uxx = uxx_match_loss(n_samples=uxx_ns)
    L_h1  = h1_penalty(n=2048)
    y_rand = model(torch.rand(2048,3,device=device)); L_reg = (y_rand**2).mean()

    A_for_theta = A_from_parts(A3_learn(), KNOWN_FIXED).detach()
    L_vprior = v_ols_prior_loss_local(d_fixed=0.005, A_for_theta=A_for_theta,
                                      Wv_ols=800.0, batch_n=12000)

    # Soft anchor of φ toward the current structured proxy (from EMA)
    with torch.no_grad():
        La_hat_ema, phi_hat_ema, fT_hat_ema = a3_proxies_from_A(globals()['A_ols_ema'])
    pA_now = A3_learn()
    phi_anchor =  (pA_now["phi"] - torch.tensor(phi_hat_ema, device=device))**2
    W_PHI_ANCH = 0.15 if phase == "P1" else (0.06 if phase == "P2" else 0.06)

    losses = dict(
        DATA=L_data, PDE=L_pde, IC=L_ic, A0=L_a0, FLUX=L_flux,
        STRUCT=W_STRUCT*L_struct, UXX=W_UXX*L_uxx, PDATA=L_pdata,
        AFOCUS=W_AFOCUS*L_afocus, RATIO=W_RATIO*L_ratio, PHIR=L_phir
    )
    tot, eff = uw(losses)
    tot = tot + 1e-7*L_reg + L_H1*L_h1 + L_vprior + W_PHI_ANCH*phi_anchor
    tot.backward()

    # Initial micro-freeze: let φ adjust more rapidly
    if ep < 600 and phase == "P1":
        # Every other iteration, freeze Λ and f_T so that φ can absorb the ratio
        if (ep % 2) == 0:
            if A3_learn.rho.grad is not None:
                  A3_learn.rho.grad[0].zero_()  # Λ
                  A3_learn.rho.grad[2].zero_()  # f_T

    scale_grads_local(v_learn, gV)
    scale_grads_local(A3_learn, gA)

    # Specific clipping on A3 to avoid large jumps
    max_step = 0.15
    with torch.no_grad():
        for p in params_A:
            if p.grad is not None:
                gnorm = p.grad.norm()
                if torch.isfinite(gnorm) and gnorm > max_step:
                    p.grad.mul_(max_step / (gnorm + 1e-12))

    params_trainables = trainable_params_from(opt)
    torch.nn.utils.clip_grad_norm_(params_trainables, 1.0)

    opt.step(); sched.step()

    if (ep % 500) == 0 or ep == 1:
        with torch.no_grad():
            v_hat = v_learn(); pA = A3_learn()
            v_ols_q = theta_ols_from_data_v_only(batch_n=4096, A_use=A_for_theta, d_fix=0.005)
            v_ols_q = abs(v_ols_q)
        dt = time.time() - t0; t0 = time.time()
        print(
            f"Ep {ep:4d} | phase={phase} "
            f"| Lpde {L_pde.item():.2e} Lpdata {L_pdata.item():.2e} "
            f"Lstruct {L_struct.item():.2e} Lafocus {L_afocus.item():.2e} Lratio {L_ratio.item():.2e} Lphir {L_phir.item():.2e} "
            f"| v={float(v_hat):.5f} (OLS {v_ols_q:.5f}) "
            f"| Λ={float(pA['Lambda']):.3f} φ={float(pA['phi']):.3f} fT={float(pA['f_T']):.3f} "
            f"| eff(PDE)={eff.get('PDE',float('nan')):.2f} eff(UXX)={eff.get('UXX',float('nan')):.2f} "
            f"eff(STRUCT)={eff.get('STRUCT',float('nan')):.2f} eff(AFOCUS)={eff.get('AFOCUS',float('nan')):.2f} eff(RATIO)={eff.get('RATIO',float('nan')):.2f} eff(PHIR)={eff.get('PHIR',float('nan')):.2f}"
        )

        def pct(h, t): return abs(h - t) / (abs(t) + 1e-12) * 100.0
        La_ok  = pct(float(pA['Lambda']), 0.35) < 10.0
        phi_ok = pct(float(pA['phi']),    0.4)  < 10.0
        fT_ok  = pct(float(pA['f_T']),    0.5)  < 10.0
        v_ok   = pct(float(v_hat),        0.06) < 10.0

        if all([La_ok, phi_ok, fT_ok, v_ok]):
            ok_count += 1
            if ep >= 4200 and ok_count >= 2:
                print("Early stop: all parameters < 10% for two consecutive checkpoints.")
                break
        else:
            ok_count = 0

## Final diagnostics and reporting

This cell reports the final inverse estimates, computes relative errors with respect to the reference values, evaluates an identifiability score, and saves a summary file.

In [ ]:
# ===========
# CELL 15 — Diagnostics & Reporting
# ===========
@torch.no_grad()
def identifiability_score(n=4000):
    t=torch.rand(n,1,device=device); a=torch.rand(n,1,device=device); x=torch.rand(n,1,device=device)
    U0 = model(torch.cat([t,a,x],1))
    eps=1.0/max(1,(len(xi_grid)-1))
    Up = model(torch.cat([t,a,(x+eps).clamp(0,1)],1))
    Um = model(torch.cat([t,a,(x-eps).clamp(0,1)],1))
    x_rng=(x_max-x_min)+1e-12
    Ux  = (Up-Um)/(2*eps)*(y_std/x_rng)
    Uxx = (Up-2*U0+Um)/(eps**2)*(y_std/(x_rng**2))
    num = Uxx.abs().mean().item()
    den = Ux.abs().mean().item()+1e-9
    return num/den

def rel_err_pct(h, t): return float(abs(h - t) / (abs(t) + 1e-12) * 100.0)

v_hat = v_learn(); pA = A3_learn()

print("\n=== Estimates (learned; d is FIXED) ===")
print(f"d_fixed  ={float(0.005):.6f} (report {0.005:.6f})")
print(f"v_hat    ={float(v_hat):.6f} (report {0.06:.6f})")
print(f"Lambda   ={float(pA['Lambda']):.6f} (report {0.35:.6f})")
print(f"phi      ={float(pA['phi']):.6f} (report {0.4:.6f})")
print(f"f_T      ={float(pA['f_T']):.6f} (report {0.5:.6f})")

print("\n=== Relative errors (%) (vs REPORT truths) ===")
print(f"v:      {rel_err_pct(float(v_hat), 0.06):6.2f}%")
print(f"Lambda: {rel_err_pct(float(pA['Lambda']), 0.35):6.2f}%")
print(f"phi:    {rel_err_pct(float(pA['phi']),    0.4):6.2f}%")
print(f"f_T:    {rel_err_pct(float(pA['f_T']),    0.5):6.2f}%")

print("\nIdentifiability (diffusion/advection) ratio ~", identifiability_score())

summary = {
    "SIGN_V": -1,
    "d_fixed": 0.005,
    "v_hat": float(v_hat),
    "Lambda_hat": float(pA["Lambda"]), "phi_hat": float(pA["phi"]), "f_T_hat": float(pA["f_T"]),
    "report_truths": {"d": 0.005, "v": 0.06, "Lambda": 0.35, "phi": 0.4, "f_T": 0.5},
}
with open("inverse_summary_d_fixed_patched.json","w") as f:
    json.dump(summary, f, indent=2)
print("\nSaved: inverse_summary_d_fixed_patched.json")